In [ ]:
""" 
In this part, we get nodes from 7 RF textbooks and filter them. And node.jsonl is the final file.

We use the following seven RF-related textbooks to construct the domain corpus: 
Microwave Engineering by David M. Pozar, 
RF and Microwave Power Amplifier Design by Andrei Grebennikov, 
RF Microelectronics by Behzad Razavi, 
RF Power Amplifiers by Marian K. Kazimierczuk, 
RF Power Amplifiers for Wireless Communications by Steve C. Cripps, 
Advanced Techniques in RF Power Amplifier Design by Steve C. Cripps,
and RF Circuit Design: Theory & Applications by Reinhold Ludwig.
"""

In [ ]:
import re, json
from pathlib import Path

CUR_DIR = Path.cwd()          
OUTPUT_FILE = CUR_DIR / "node0.jsonl"

re_img_md  = re.compile(r'!\[.*?\]\(.*?\)')           
re_figure  = re.compile(r'^\s*(Figure|Fig\.)\s*\d+', re.I)
re_subheading = re.compile(r'^(#+\s*\d+\.\d+(?:\.\d+)*\b.*)$', re.MULTILINE)

def process_md_file(path: Path):
    with path.open(encoding="utf-8") as f:
        content = f.read()

    lines = content.split('\n')
    output_lines = []
    in_section = False
    current_section = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue

        if re_subheading.match(line):
            if current_section:
                section_content = '\n'.join(current_section)
                cleaned_content = clean_content_simple(section_content)
                if cleaned_content:
                    output_lines.append(cleaned_content)
                current_section = []

            in_section = True
            current_section.append(f"<little_title>{line}")
        else:

            if in_section:
                current_section.append(line)

    if current_section:
        section_content = '\n'.join(current_section)
        cleaned_content = clean_content_simple(section_content)
        if cleaned_content:
            output_lines.append(cleaned_content)

    if not output_lines:
        cleaned_content = clean_content_simple(content)
        if cleaned_content:
            yield cleaned_content
    else:
        for content_text in output_lines:
            yield content_text

def clean_content_simple(text: str) -> str:
    if not text:
        return ""
    
    lines = []
    for line in text.split('\n'):
        line = re_img_md.sub('', line).strip()
        if not line:
            continue
        if re_figure.match(line):
            continue
        lines.append(line)
    
    return '\n'.join(lines)

with OUTPUT_FILE.open("w", encoding="utf-8") as fout:
    for md_path in CUR_DIR.glob("*.md"):
        print(f"deal with file: {md_path.name}")
        for para in process_md_file(md_path):
            fout.write(json.dumps({"text": para}, ensure_ascii=False) + "\n")

print(f"output dataset is: {OUTPUT_FILE}")

In [ ]:
import re, json
from pathlib import Path

CUR_DIR = Path.cwd()
SPECIAL_FILE = "markdown+(11).md"
NODE_FILE = "node0.jsonl"
MERGED_FILE = "node1.jsonl"

re_img_md  = re.compile(r'!\[.*?\]\(.*?\)')           
re_figure  = re.compile(r'^\s*(Figure|Fig\.)\s*\d+', re.I)
re_heading = re.compile(r'^(#+ .*)$', re.MULTILINE)

def process_special_file(path: Path):
    with path.open(encoding="utf-8") as f:
        content = f.read()

    lines = content.split('\n')
    output_lines = []
    current_section = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue

        if re_heading.match(line):
            if current_section:
                section_content = '\n'.join(current_section)
                cleaned_content = clean_content_simple(section_content)
                if cleaned_content:
                    output_lines.append(cleaned_content)
                current_section = []
            current_section.append(line)
        else:
            if current_section:
                current_section.append(line)

    if current_section:
        section_content = '\n'.join(current_section)
        cleaned_content = clean_content_simple(section_content)
        if cleaned_content:
            output_lines.append(cleaned_content)

    return output_lines

def clean_content_simple(text: str) -> str:
    if not text:
        return ""
    
    lines = []
    for line in text.split('\n'):
        line = re_img_md.sub('', line).strip()
        if not line or re_figure.match(line):
            continue
        lines.append(line)
    
    return '\n'.join(lines)

node_content = []
if Path(NODE_FILE).exists():
    with open(NODE_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            node_content.append(data["text"])

special_path = Path(SPECIAL_FILE)
if special_path.exists():
    special_content = process_special_file(special_path)
    all_content = node_content + special_content

    with open(MERGED_FILE, "w", encoding="utf-8") as fout:
        for content in all_content:
            fout.write(json.dumps({"text": content}, ensure_ascii=False) + "\n")
print(f"output dataset is: {MERGED_FILE}")

In [ ]:
import json
import re
from pathlib import Path

INPUT_FILE = Path("node1.jsonl")
OUTPUT_FILE = Path("node2.jsonl")

# filter the paragraphs which only have simple title
def filter_single_title_nodes():
    filtered_count = 0
    total_count = 0
    useful_count = 0
    
    with INPUT_FILE.open("r", encoding="utf-8") as fin, \
         OUTPUT_FILE.open("w", encoding="utf-8") as fout:
        
        for line in fin:
            total_count += 1
            data = json.loads(line.strip())
            text = data["text"]

            if has_actual_content(text):
                cleaned_text = remove_little_title_tags(text)
                useful_count += 1
                fout.write(json.dumps({
                    "id": f"{useful_count}",  
                    "text": cleaned_text
                }, ensure_ascii=False) + "\n")
            else:
                filtered_count += 1
    
    print(f"formal nodes: {total_count}")
    print(f"useful nodes: {useful_count}")
    print(f"filtered nodes: {filtered_count}")
    print(f"output file: {OUTPUT_FILE}")

def has_actual_content(text: str) -> bool:
    lines = text.split('\n')
    if len(lines) <= 1:
        return False
    for line in lines[1:]:
        if line.strip() and not line.startswith('<little_title>'):
            return True
    return False

def remove_little_title_tags(text: str) -> str:
    return re.sub(r'<little_title>', '', text)

if __name__ == "__main__":
    filter_single_title_nodes()

In [ ]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("Qwen-0.6B-local", trust_remote_code=True)

token_counts = []
with open("node2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_counts.append(len(tokens))

print(f"total paragraphs: {len(token_counts)}")
print(f"average tokens in each paragraph: {np.mean(token_counts):.1f}")

In [ ]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

token_counts = []
with open("node2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_counts.append(len(tokens))

print(f"total paragraphs: {len(token_counts)}")
print(f"average tokens in each paragraph: {np.mean(token_counts):.1f}")

In [ ]:
import json

def reassign_sequential_ids(input_file, output_file, start_id=1, id_key='id'):
    current_id = start_id
    
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        for line in f_in:
            if line.strip():  
                data = json.loads(line.strip())
         
                data[id_key] = current_id
                current_id += 1

                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
    
    print(f"new id: {start_id} - {current_id - 1}")

if __name__ == "__main__":
    input_file = "node3.jsonl"    
    output_file = "node4.jsonl" 
    
    reassign_sequential_ids(input_file, output_file)
    print(f"Saved new file as {output_file}")

In [ ]:
import json

def reassign_sequential_ids(input_file, output_file, start_id=1, id_key='id'):
    current_id = start_id
    
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        for line in f_in:
            if line.strip():  
                data = json.loads(line.strip())
         
                data[id_key] = str(current_id)
                current_id += 1

                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
    
    print(f"new id: {start_id} - {current_id - 1}")

if __name__ == "__main__":
    input_file = "node6.jsonl"    
    output_file = "node7.jsonl" 
    
    reassign_sequential_ids(input_file, output_file)
    print(f"Saved new file as {output_file}")

In [ ]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD = 10000  

token_counts = []
long_paragraphs = []  

with open("node2.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD} tokens: {len(long_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")


In [ ]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD = 10000  

token_counts = []
long_paragraphs = []  

with open("node4.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD} tokens: {len(long_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

In [ ]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD_LONG = 10000  
THRESHOLD_SHORT = 150

token_counts = []
long_paragraphs = []  
short_paragraphs = []

with open("node4.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD_LONG:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })
        
        if token_count < THRESHOLD_SHORT:
            short_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD_LONG} tokens: {len(long_paragraphs)}")
print(f"Paragraphs below {THRESHOLD_SHORT} tokens: {len(short_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

if short_paragraphs:
    print("\nDetails of short paragraphs:")
    for para in short_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

In [ ]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD_LONG = 10000  
THRESHOLD_SHORT = 150

token_counts = []
long_paragraphs = []  
short_paragraphs = []

with open("node6.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD_LONG:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })
        
        if token_count < THRESHOLD_SHORT:
            short_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD_LONG} tokens: {len(long_paragraphs)}")
print(f"Paragraphs below {THRESHOLD_SHORT} tokens: {len(short_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

if short_paragraphs:
    print("\nDetails of short paragraphs:")
    for para in short_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")